# **Data Preprocessing**
---

This notebook prepares the CITE-seq PBMC dataset (Hao et al. (2021) Cell) from CellxGene for Geneformer fine-tuning.

## Why Preprocessing?

Geneformer requires specific data formatting:
- **Raw counts**: Models raw transcript counts, not normalized data
- **Ensembl IDs**: Uses Ensembl gene IDs without version suffixes
- **Required metadata**: Needs specific fields in `.obs` and `.var`

## Preprocessing Steps

1. **Load raw data** from CellxGene h5ad format
2. **Extract raw counts** from `adata.raw.X` or `adata.layers['counts']`
3. **Clean Ensembl IDs** by removing version suffixes (e.g., `ENSG00000000003.15` → `ENSG00000000003`)
4. **Add required fields**:
   - `n_counts`: Total transcript counts per cell
   - `joinid`: Unique cell identifiers
   - `ensembl_id`: Gene identifiers in `.var`
5. **Save processed data** ready for tokenization

## Importance for Geneformer

**Why raw counts?**
- Geneformer learns from the distribution of raw transcript abundances
- Normalization can remove biological signal that the model needs
- Rank-based tokenization requires integer counts

**Why Ensembl IDs?**
- Geneformer's vocabulary is based on Ensembl gene IDs
- Version suffixes would create mismatches with the pretrained model
- Ensures compatibility with Geneformer's token dictionary

**Critical for downstream tasks:**
- Proper preprocessing ensures the model can tokenize your data
- Maintains compatibility with pretrained Geneformer weights
- Enables accurate cell type classification fine-tuning
---

## Setup Instructions

Before running this notebook, update the paths in the configuration cell below:
- `BASE_DIR`: Root directory for the benchmarking repository
- `RAW_DATA_DIR`: Directory containing raw h5ad files
- `PREPROCESSED_DIR`: Output directory for preprocessed files

In [1]:
import os
from pathlib import Path

# ===== CONFIGURATION - Update these paths for your environment =====
BASE_DIR = Path("/scratch/tmurugan/geneformer_benchmarking")
RAW_DATA_DIR = BASE_DIR / "data" / "raw_data"
PREPROCESSED_DIR = BASE_DIR / "data" / "preprocessed_data"

# Input/output files
RAW_FILE = "cellxgene_citeseq_pbmc.h5ad"
OUTPUT_FILE = "citeseq_pbmc_geneformer_processed.h5ad"

# Create directories if they don't exist
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration loaded:")
print(f"  Base directory: {BASE_DIR}")
print(f"  Raw data: {RAW_DATA_DIR}")
print(f"  Preprocessed output: {PREPROCESSED_DIR}")

Configuration loaded:
  Base directory: /scratch/tmurugan/geneformer_benchmarking
  Raw data: /scratch/tmurugan/geneformer_benchmarking/data/raw_data
  Preprocessed output: /scratch/tmurugan/geneformer_benchmarking/data/preprocessed_data


## Load Raw Data

In [2]:
# Import libraries
import scanpy as sc
import numpy as np
import scipy.sparse as sp

# Load anndata using configured paths
raw_path = RAW_DATA_DIR / RAW_FILE
output_path = PREPROCESSED_DIR / OUTPUT_FILE

adata = sc.read_h5ad(raw_path)  
print(adata)

AnnData object with n_obs × n_vars = 161764 × 20264
    obs: 'nCount_ADT', 'nFeature_ADT', 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'lane', 'donor_id', 'time', 'celltype.l1', 'celltype.l2', 'celltype.l3', 'Phase', 'nCount_SCT', 'nFeature_SCT', 'cell_type_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'celltype.l3_colors', 'citation', 'default_embedding', 'neighbors', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_apca', 'X_aumap', 'X_pca', 'X_spca', 'X_umap', 'X_wnn.umap'
    va

## Extract Raw Counts and Clean Ensembl IDs

In [3]:
# Ensure Ensembl IDs without version suffix 
adata.var_names = [str(g).split('.')[0] for g in adata.var_names]

# Move raw counts into adata.X (required for geneformer)
if adata.raw is not None:
    print("Using adata.raw.X as raw counts")
    adata.X = adata.raw.X.copy()
elif "counts" in adata.layers:
    print("Using adata.layers['counts'] as raw counts")
    adata.X = adata.layers["counts"].copy()
else:
    raise RuntimeError("No raw counts found in this dataset for Geneformer.")

Using adata.raw.X as raw counts


## Add Required Fields

In [4]:
# Remove raw to avoid confusion
adata.raw = None

# Compute n_counts from adata.X 
adata.obs["n_counts"] = np.array(adata.X.sum(axis=1)).ravel().astype(int)

# Create obs and var fields
adata.obs["joinid"] = np.arange(adata.n_obs).astype(int) #optional
adata.var["ensembl_id"] = adata.var.index.astype(str) # genes have to be represented by ENSEMBL ID for geneformer

## Save Processed Data

In [5]:
# Save cleaned file
adata.write(output_path, compression='gzip')

print(
    "Wrote Geneformer-ready h5ad:",
    "cells =", adata.n_obs,
    ", genes =", adata.n_vars
)

Wrote Geneformer-ready h5ad: cells = 161764 , genes = 20264
